# Submissão 3 — Modelo A (Stacking Ensemble: RoBERTa-base + DeBERTa-v3-base)

In [2]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import (
    RobertaTokenizer, RobertaForSequenceClassification,
    DebertaV2Tokenizer, DebertaV2ForSequenceClassification
)
from sklearn.metrics import f1_score
from transformer_utils import InferenceDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

LABEL2ID     = {'google': 0, 'anthropic': 1, 'meta': 2, 'openai': 3, 'human': 4}
ID2LABEL     = {v: k for k, v in LABEL2ID.items()}
ID2LABEL_OUT = {0: 'Google', 1: 'Anthropic', 2: 'Meta', 3: 'OpenAI', 4: 'Human'}

device: cuda


## *carregar modelos*

In [3]:
roberta_path  = '../models/model_roberta'
roberta_tok   = RobertaTokenizer.from_pretrained(roberta_path)
roberta_model = RobertaForSequenceClassification.from_pretrained(roberta_path).to(device)
roberta_model.eval()
print('RoBERTa-base carregado')

deberta_path  = '../models/model_deberta'
deberta_tok   = DebertaV2Tokenizer.from_pretrained(deberta_path)
deberta_model = DebertaV2ForSequenceClassification.from_pretrained(deberta_path).float().to(device)
deberta_model.eval()
print('DeBERTa-v3-base carregado')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RoBERTa-base carregado


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DeBERTa-v3-base carregado


## *extrair probabilidades*

In [4]:
MAX_LEN = 128

def get_probabilities(model, tokenizer, texts, batch_size=32):
    ds     = InferenceDataset(texts, tokenizer, MAX_LEN)
    loader = DataLoader(ds, batch_size=batch_size)
    all_probs = []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            probs  = F.softmax(logits, dim=-1).cpu().numpy()
            all_probs.append(probs)
    return np.vstack(all_probs)

# dataset-exemplos (para encontrar o melhor peso)
df_val   = pd.read_csv('../datasets/dataset-exemplos.csv', sep=';')
val_texts = df_val['Text'].fillna('').tolist()
val_y     = np.array([LABEL2ID[l.strip().lower()] for l in df_val['Label']])

# subm3 (para submissão)
df_subm  = pd.read_csv('subm3.csv', sep=';')
subm_texts = df_subm['Text'].fillna('').tolist()

print('a extrair probabilidades do RoBERTa...')
probs_r_val  = get_probabilities(roberta_model, roberta_tok, val_texts)
probs_r_subm = get_probabilities(roberta_model, roberta_tok, subm_texts)

print('a extrair probabilidades do DeBERTa...')
probs_d_val  = get_probabilities(deberta_model, deberta_tok, val_texts)
probs_d_subm = get_probabilities(deberta_model, deberta_tok, subm_texts)

print('feito.')

a extrair probabilidades do RoBERTa...
a extrair probabilidades do DeBERTa...
feito.


## *sweep de pesos no dataset-exemplos*

In [5]:
print(f'  alpha  |   acc   |  F1-macro')
print('-' * 35)

best_f1    = 0.0
best_alpha = 0.5

for alpha in np.arange(0.0, 1.01, 0.05):
    probs = alpha * probs_r_val + (1 - alpha) * probs_d_val
    preds = probs.argmax(axis=1)
    acc   = (preds == val_y).mean()
    f1    = f1_score(val_y, preds, average='macro')
    marker = ' <-- best' if f1 > best_f1 else ''
    if f1 > best_f1:
        best_f1    = f1
        best_alpha = alpha
    print(f'  {alpha:.2f}   |  {acc:.4f} |  {f1:.4f}{marker}')

print(f'\nmelhor alpha={best_alpha:.2f} (RoBERTa) | F1={best_f1:.4f}')

  alpha  |   acc   |  F1-macro
-----------------------------------
  0.00   |  0.6960 |  0.6060 <-- best
  0.05   |  0.6960 |  0.6060
  0.10   |  0.6960 |  0.6060
  0.15   |  0.7040 |  0.6221 <-- best
  0.20   |  0.7040 |  0.6221
  0.25   |  0.7040 |  0.6221
  0.30   |  0.7120 |  0.6391 <-- best
  0.35   |  0.7200 |  0.6436 <-- best
  0.40   |  0.7200 |  0.6436
  0.45   |  0.7200 |  0.6436
  0.50   |  0.7360 |  0.6533 <-- best
  0.55   |  0.7600 |  0.6956 <-- best
  0.60   |  0.7680 |  0.7116 <-- best
  0.65   |  0.7600 |  0.6990
  0.70   |  0.7520 |  0.6932
  0.75   |  0.7600 |  0.7031
  0.80   |  0.7680 |  0.7058
  0.85   |  0.7680 |  0.7058
  0.90   |  0.7760 |  0.7186 <-- best
  0.95   |  0.7760 |  0.7186
  1.00   |  0.7760 |  0.7186

melhor alpha=0.90 (RoBERTa) | F1=0.7186


## *aplicar ao subm3 e guardar*

In [6]:
probs_final = best_alpha * probs_r_subm + (1 - best_alpha) * probs_d_subm
preds_final = probs_final.argmax(axis=1)

df_subm['Label'] = [ID2LABEL_OUT[p] for p in preds_final]
print(df_subm['Label'].value_counts())

filename = 'subm3-g2-MEI-A.csv'
df_subm[['ID', 'Label']].to_csv(filename, index=False, sep=';')
print(f"\nFicheiro '{filename}' guardado (alpha={best_alpha:.2f})")

Label
Meta         48
Human        45
Google       22
Anthropic    18
OpenAI       17
Name: count, dtype: int64

Ficheiro 'subm3-g2-MEI-A.csv' guardado (alpha=0.90)
